# ETL-20 · dim_perfil_vitima e dim_transporte

- **Etapa:** ETL-20, dim_perfil_vitima e dim_transporte
- **Responsável:** Pedro Wellington
- **Issue:** ETL-20
- **Entrada:** `data/intermediarios/05_nulos_categorias.parquet`
- **Saída:** `data/intermediarios/dims/dim_perfil_vitima.parquet` e `dim_transporte.parquet`

Cria as duas dimensões de combinações do modelo estrela, cada uma com chave substituta e os membros especiais -1 e -2.

## 1. Setup

In [18]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

INTERM = RAIZ / "data" / "intermediarios"      # .parquet (o .gitignore já ignora)
DIMS   = INTERM / "dims"                       # saída: uma pasta só para as dimensões
DIMS.mkdir(parents=True, exist_ok=True)

ENTRADA = INTERM / "05_nulos_categorias.parquet"   # saída da ETL-16

## 2. Carregar a entrada

In [19]:
# colunas de cada dimensão (nomes iguais aos do modelo estrela)
COLUNAS_TRANSPORTE = ['tipo_de_vitima', 'modo_transporte_vitima', 'tipo_veiculo_vitima']
COLUNAS_PERFIL = ['sexo', 'faixa_etaria_demografica', 'faixa_etaria_legal',
                  'nacionalidade', 'grau_de_instrucao']

# lê só as 8 colunas que importam (o parquet tem 34)
df = pd.read_parquet(ENTRADA, columns=COLUNAS_TRANSPORTE + COLUNAS_PERFIL)
print('shape:', df.shape); df.head()

shape: (806977, 8)


,tipo_de_vitima,modo_transporte_vitima,tipo_veiculo_vitima,sexo,faixa_etaria_demografica,faixa_etaria_legal,nacionalidade,grau_de_instrucao
0,PASSAGEIRO,AUTOMOVEL,AUTOMOVEL,FEMININO,30 a 34,30-34,BRASILEIRA,MÉDIO
1,CONDUTOR,AUTOMOVEL,AUTOMOVEL,MASCULINO,30 a 34,30-34,BRASILEIRA,NÃO INFORMADO
2,PEDESTRE,NÃO SE APLICA,NÃO SE APLICA,FEMININO,65 a 69,65-69,BRASILEIRA,SUPERIOR
3,CONDUTOR,AUTOMOVEL,AUTOMOVEL,MASCULINO,25 a 29,25-29,BRASILEIRA,NÃO INFORMADO
4,PASSAGEIRO,NÃO INFORMADO,NÃO INFORMADO,MASCULINO,65 a 69,65-69,BRASILEIRA,SUPERIOR


## 3. Diagnóstico ANTES

In [20]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
tipo_de_vitima,str,0.0,4
modo_transporte_vitima,str,0.0,9
tipo_veiculo_vitima,str,0.0,8
sexo,str,0.0,3
faixa_etaria_demografica,str,0.0,20
faixa_etaria_legal,str,0.0,15
nacionalidade,str,0.0,91
grau_de_instrucao,str,0.0,4


## 4. Criar as dimensões

Cada dimensão guarda uma linha por combinação que existe nos dados, com uma chave `sk_` e os membros -1 e -2.

### 4.1 Função que monta a dimensão

In [21]:
def criar_dimensao(df: pd.DataFrame, colunas: list, nome_sk: str) -> pd.DataFrame:
    """Uma linha por combinação existente, com chave sk_ e os membros -1 e -2."""
    # 1. combinações distintas das colunas, em ordem
    dim = df[colunas].drop_duplicates().sort_values(colunas).reset_index(drop=True)

    # 2. combinações que são só "NÃO INFORMADO" ou só "NÃO SE APLICA" não viram linha:
    #    elas ficam representadas pelos membros -1 e -2
    so_nao_informado = (dim[colunas] == 'NÃO INFORMADO').all(axis=1)
    so_nao_se_aplica = (dim[colunas] == 'NÃO SE APLICA').all(axis=1)
    dim = dim[~(so_nao_informado | so_nao_se_aplica)].reset_index(drop=True)

    # 3. chave substituta: inteiro de 1 em diante
    dim.insert(0, nome_sk, range(1, len(dim) + 1))

    # 4. membros especiais -1 e -2
    especiais = pd.DataFrame({nome_sk: [-1, -2]})
    for c in colunas:
        especiais[c] = ['NÃO INFORMADO', 'NÃO SE APLICA']

    return pd.concat([especiais, dim], ignore_index=True)

### 4.2 dim_transporte


In [22]:
dim_transporte = criar_dimensao(df, COLUNAS_TRANSPORTE, 'sk_transporte')
dim_transporte

,sk_transporte,tipo_de_vitima,modo_transporte_vitima,tipo_veiculo_vitima
0,-1,NÃO INFORMADO,NÃO INFORMADO,NÃO INFORMADO
1,-2,NÃO SE APLICA,NÃO SE APLICA,NÃO SE APLICA
2,1,CONDUTOR,AUTOMOVEL,AUTOMOVEL
3,2,CONDUTOR,BICICLETA,BICICLETA
4,3,CONDUTOR,CAMINHAO,CAMINHAO
5,4,CONDUTOR,MOTOCICLETA,MOTOCICLETA
6,5,CONDUTOR,NÃO INFORMADO,BICICLETA
7,6,CONDUTOR,NÃO INFORMADO,NÃO INFORMADO
8,7,CONDUTOR,ONIBUS,ONIBUS
9,8,CONDUTOR,OUTROS,OUTROS


### 4.3 dim_perfil_vitima


In [23]:
dim_perfil_vitima = criar_dimensao(df, COLUNAS_PERFIL, 'sk_perfil_vitima')
print(len(dim_perfil_vitima), 'linhas')
dim_perfil_vitima.head(10)

1500 linhas


,sk_perfil_vitima,sexo,faixa_etaria_demografica,faixa_etaria_legal,nacionalidade,grau_de_instrucao
0,-1,NÃO INFORMADO,NÃO INFORMADO,NÃO INFORMADO,NÃO INFORMADO,NÃO INFORMADO
1,-2,NÃO SE APLICA,NÃO SE APLICA,NÃO SE APLICA,NÃO SE APLICA,NÃO SE APLICA
2,1,FEMININO,00 a 04,0-17,BOLIVIANA,SUPERIOR
3,2,FEMININO,00 a 04,0-17,BRASILEIRA,BÁSICO
4,3,FEMININO,00 a 04,0-17,BRASILEIRA,MÉDIO
5,4,FEMININO,00 a 04,0-17,BRASILEIRA,NÃO INFORMADO
6,5,FEMININO,00 a 04,0-17,BRASILEIRA,SUPERIOR
7,6,FEMININO,00 a 04,0-17,COLOMBIANA,SUPERIOR
8,7,FEMININO,00 a 04,0-17,HAITIANA,SUPERIOR
9,8,FEMININO,00 a 04,0-17,NÃO INFORMADO,NÃO INFORMADO


## 5. Conferir e salvar

In [24]:
tabelas = [
    ('dim_transporte', dim_transporte, COLUNAS_TRANSPORTE, 'sk_transporte'),
    ('dim_perfil_vitima', dim_perfil_vitima, COLUNAS_PERFIL, 'sk_perfil_vitima'),
]

for nome, dim, colunas, sk in tabelas:
    # a chave não pode se repetir e nenhuma combinação pode aparecer duas vezes
    print(nome, '|', len(dim), 'linhas | sk única:', dim[sk].is_unique,
          '| combinações repetidas:', dim.duplicated(subset=colunas).sum())
    # salva em dims/
    dim.to_parquet(DIMS / f'{nome}.parquet', index=False)

dim_transporte | 25 linhas | sk única: True | combinações repetidas: 0
dim_perfil_vitima | 1500 linhas | sk única: True | combinações repetidas: 0


In [25]:
# cada pessoa da base tem que achar uma chave na dimensão
for nome, dim, colunas, sk in tabelas:
    m = df[colunas].merge(dim, on=colunas, how='left')
    print(nome, '| linhas sem chave:', int(m[sk].isna().sum()), '| total:', len(m))

dim_transporte | linhas sem chave: 0 | total: 806977
dim_perfil_vitima | linhas sem chave: 0 | total: 806977


## 6. Resumo

**O que foi feito:** criação das duas dimensões de combinações do modelo estrela, a partir da saída da ETL-16. Cada uma tem uma linha por combinação que existe nos dados, uma chave substituta `sk_` e os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA). Os arquivos foram salvos em `data/intermediarios/dims/`.

**Resultado:**
- `dim_transporte`: 25 linhas (23 combinações de tipo de vítima, modo de transporte e tipo de veículo)
- `dim_perfil_vitima`: 1.500 linhas (1.498 combinações de sexo, faixas etárias, nacionalidade e grau de instrução)

**O que foi conferido:** as chaves são únicas, nenhuma combinação se repete e as 806.977 pessoas encontram uma chave nas duas dimensões.

**Observação:** combinações em que todas as colunas são NÃO INFORMADO não viram linha própria, elas usam o membro -1. A idade exata fica na tabela fato, como no modelo.

**Observação 2:** `nacionalidade` ainda pode ter espaços no fim do texto, que são tratados na ETL-14. Quando essa etapa entrar, o número de combinações pode diminuir um pouco.